In [1]:
## Install the dependencies
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Set paths and check hardware

In [4]:
import csv
import json
import shutil
import subprocess
import unicodedata
from pathlib import Path

import jiwer
import numpy as np
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from torch.utils.data import Dataset 
from tqdm.auto import tqdm
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperForConditionalGeneration, WhisperProcessor)

ROOT = Path.cwd().resolve()
if ROOT.name == "../myanmar-asr":
    ROOT = ROOT.parent

assert (ROOT / "dataset" / "NLP Project1" / "audio_files").is_dir(), f"Wrong Jupyter working directory: {ROOT}"
assert (ROOT / "dataset" / "asr_work" / "train_whisper.csv").is_file(), f"Missing prepared manifests" 
assert shutil.which("ffmpeg"), "Installing FFmpeg and restart Jupyter so ffmpeg is on PATH" 

ModuleNotFoundError: No module named 'jiwer'

In [ ]:
QUICK_RUN = False
MODEL_ID = "openai/whisper-small" 
RUN_DIR = ROOT / "asr_work" / ("ucsy_whisper_smoke" if QUICK_RUN else "ucsy_whisper_full")
CACHE_DIR = ROOT / "asr_work" / "ucsy_audio_16k" 
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
print({
    "root": str(ROOT),
    "device": device,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "ffmpeg": shutil.which("ffmpeg")
})

if device == "cpu":
    print("CPU can check the data pipeline, but full Whisper needs a GPU in practice.")

NameError: name 'torch' is not defined

### Load UCSY-only splits and checking leakage

In [ ]:
def load_ucsy_split(name):
    with (ROOT / "asr_work" / f"{name}_whisper.csv").open(newline="", encoding="utf-8") as handle:
        rows = [row for row in csv.DictReader(handle) if row["source"] == "ucsy"]
    assert rows, f"No UCSY rows in {name}"

    for row in rows:
        path = (ROOT / row["audio"]).resolve()
        assert path.is_relative_to((ROOT / "NLP Project1" / "audio_files").resolve()), row["audio"]
        assert path.is_file() and path.stat().st_size > 0, row["audio"]
        assert 0 < float(row["duration_seconds"]) < 30, row["sample_id"]